In [1]:
import os 

In [2]:
%pwd

'd:\\MLops\\DataScienceProject_1\\Health_premium_calculator\\research'

In [3]:
os.chdir("../")

In [25]:
%pwd

'd:\\MLops\\DataScienceProject_1\\Health_premium_calculator'

In [17]:
import pandas as pd

data=pd.read_csv("artifacts/data_ingestion/data.csv")
data.head()

,Age,Gender,Region,Marital_status,Number Of Dependants,BMI_Category,Smoking_Status,Employment_Status,Income_Level,Income_Lakhs,Medical History,Insurance_Plan,Annual_Premium_Amount
0,26,Male,Northwest,Unmarried,0,Normal,No Smoking,Salaried,<10L,6,Diabetes,Bronze,9053
1,29,Female,Southeast,Married,2,Obesity,Regular,Salaried,<10L,6,Diabetes,Bronze,16339
2,49,Female,Northeast,Married,2,Normal,No Smoking,Self-Employed,10L - 25L,20,High blood pressure,Silver,18164
3,30,Female,Southeast,Married,3,Normal,No Smoking,Salaried,> 40L,77,No Disease,Gold,20303
4,18,Male,Northeast,Unmarried,0,Overweight,Regular,Self-Employed,> 40L,99,High blood pressure,Silver,13365


In [18]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Age                    50000 non-null  int64 
 1   Gender                 50000 non-null  object
 2   Region                 50000 non-null  object
 3   Marital_status         50000 non-null  object
 4   Number Of Dependants   50000 non-null  int64 
 5   BMI_Category           50000 non-null  object
 6   Smoking_Status         49989 non-null  object
 7   Employment_Status      49998 non-null  object
 8   Income_Level           49987 non-null  object
 9   Income_Lakhs           50000 non-null  int64 
 10  Medical History        50000 non-null  object
 11  Insurance_Plan         50000 non-null  object
 12  Annual_Premium_Amount  50000 non-null  int64 
dtypes: int64(4), object(9)
memory usage: 5.0+ MB


In [19]:
data.describe()

,Age,Number Of Dependants,Income_Lakhs,Annual_Premium_Amount
count,50000.000000,50000.000000,50000.000000,50000.000000
mean,34.593480,1.712080,23.018200,15768.116320
std,15.000437,1.498248,24.219197,8419.839675
min,18.000000,-3.000000,1.000000,3501.000000
25%,22.000000,0.000000,7.000000,8608.000000
50%,31.000000,2.000000,17.000000,13929.000000
75%,45.000000,3.000000,31.000000,22275.250000
max,356.000000,5.000000,930.000000,43471.000000


In [20]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataValidationConfig:
    root_dir:Path
    STATUS_FILE:str
    unzip_data_dir:Path
    all_schema:dict

In [21]:
from src.datascience.constants import *
from src.datascience.utils.common import read_yaml, create_directories

In [22]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_validation_config(self) -> DataValidationConfig:
        config = self.config.data_validation
        schema = self.schema.COLUMNS

        create_directories([config.root_dir])

        data_validation_config = DataValidationConfig(
            root_dir=config.root_dir,
            STATUS_FILE=config.STATUS_FILE,
            unzip_data_dir = config.unzip_data_dir,
            all_schema=schema,
        )

        return data_validation_config

In [ ]:
import os
from src.datascience import logger

In [37]:
class DataValidation:
    def __init__(self, config: DataValidationConfig):
        self.config = config

    def validate_all_columns(self)-> bool:
        try:
            validation_status = None

            data = pd.read_csv(self.config.unzip_data_dir)
            all_cols = list(data.columns)

            all_schema = self.config.all_schema.keys()

            for col in all_cols:
                if col not in all_schema:
                    validation_status = False
                    with open(self.config.STATUS_FILE, 'w') as f:
                        f.write(f"Validation of Columns status: {validation_status}\n")
                else:
                    validation_status = True
                    with open(self.config.STATUS_FILE, 'w') as f:
                        f.write(f"Validation of Columns status: {validation_status}\n")

            return validation_status
        
        except Exception as e:
            raise e



    def validate_column_dtypes(self) -> bool:
        try:
            data = pd.read_csv(self.config.unzip_data_dir)
            validation_status = True

            for col, expected_dtype in self.config.all_schema.items():
                if col in data.columns:
                    if str(data[col].dtype) != expected_dtype:
                        validation_status = False
                        logger.warning(f"Column {col} has incorrect dtype: found {data[col].dtype}, expected {expected_dtype}")

            with open(self.config.STATUS_FILE, 'a') as f:
                f.write(f"Column Dtype Validation Status: {validation_status}\n")

            logger.info("Dtype validation complete.")
            return validation_status

        except Exception as e:
            raise e

    def check_null_values(self) -> bool:
        try:
            data = pd.read_csv(self.config.unzip_data_dir)
            null_summary = data.isnull().sum().to_dict()
            has_nulls = any(val > 0 for val in null_summary.values())

            with open(self.config.STATUS_FILE, 'a') as f:
                f.write(f"Null Value Check: {'Fail' if has_nulls else 'Pass'}\n")
                f.write(f"Nulls Summary: {null_summary}\n")

            logger.info("Null value check complete.")
            return True

        except Exception as e:
            raise e

    def check_duplicates(self) -> bool:
        try:
            data = pd.read_csv(self.config.unzip_data_dir)
            num_duplicates = data.duplicated().sum()
            has_duplicates = num_duplicates > 0

            with open(self.config.STATUS_FILE, 'a') as f:
                f.write(f"Duplicate Check: {'Fail' if has_duplicates else 'Pass'} ({num_duplicates} duplicates)\n")

            logger.info("Duplicate check complete.")
            return not has_duplicates

        except Exception as e:
            raise e


          





In [38]:
try:
    config = ConfigurationManager()
    data_validation_config = config.get_data_validation_config()
    data_validation = DataValidation(config=data_validation_config)
    data_validation.validate_all_columns()
    data_validation.validate_column_dtypes()
    data_validation.check_null_values()
    data_validation.check_duplicates()

except Exception as e:
    raise e

[2025-04-30 13:26:21,594:INFO:common:yaml file: config\config.yaml loaded successfully]
[2025-04-30 13:26:21,597:INFO:common:yaml file: params.yaml loaded successfully]
[2025-04-30 13:26:21,601:INFO:common:yaml file: schema.yaml loaded successfully]
artifacts already exists, skipping.
[2025-04-30 13:26:21,756:INFO:3308422800:Dtype validation complete.]
[2025-04-30 13:26:21,837:INFO:3308422800:Null value check complete.]
[2025-04-30 13:26:21,929:INFO:3308422800:Duplicate check complete.]
